# Шаблонные методы генерации упражнений (v2)

Реализация трёх шаблонных бейзлайнов в едином контракте JSON, совместимом с выходом дообученных LLM.
Цель — методологически честное сравнение шаблонных методов с нейросетевыми на валидационной выборке `val_300`.

## Архитектура

В отличие от прежней реализации, где каждая запись датасета EXPECT превращалась в одно упражнение с одним заданием, новая архитектура имитирует продуктовый сценарий платформы SAYIT: на вход подаётся **профиль ошибок ученика** (набор из 8–12 записей одной грамматической категории), на выходе формируется **одно упражнение с 8–12 заданиями**. Это соответствует тому, как используются дообученные LLM, и позволяет сравнивать методы по единому набору метрик Task-Relevance.

Реализованы два шаблонных метода:

* **Fill-in-the-Blanks v2** — упражнения формата `grammar_choice` и `vocabulary_fill`. Из профиля ошибок собираются 8–12 предложений, в каждом ставится пропуск на месте корректирующего слова, дистракторы подбираются по части речи, определённой в синтаксическом контексте предложения (POS-теггинг spaCy на полном предложении, а не на изолированном токене).
* **Sentence Reconstruction v2** — упражнение формата `transformation`. Жёсткое ограничение по длине (прежний фильтр `4 ≤ len ≤ 20`) снято: остаётся только нижняя граница в 4 токена. Предложения любой длины обрабатываются через chunking — разбиение по пунктуации (запятые, точки с запятой, двоеточия, тире) на 2–6 синтаксических фрагментов, которые перемешиваются как неделимые блоки. Если внутренней пунктуации нет, а длина больше 14 токенов, применяется разбиение по длине (chunks по 6 токенов). Для коротких предложений без пунктуации (4–14 токенов) перемешиваются отдельные токены при фиксированных первом и последнем значимых словах. Это даёт покрытие около 95% против 39% в прежней версии.

## Единый контракт выхода

Оба метода возвращают объект JSON со следующей структурой:

```json
{
  "target_error_category": "Preposition",
  "corrected_sentence": "...",
  "task": {
    "type": "grammar_choice",
    "instruction_en": "Choose the correct preposition for each sentence.",
    "content_en": {
      "items": [
        {"question_en": "... ___ ...", "options_en": ["of", "in", "on", "at"], "student_answer_en": "... **of** ..."},
        ...
      ]
    }
  }
}
```

Поле `student_answer_en` содержит сгенерированный ответ в той же форме, что и у дообученных LLM (правильный токен или восстановленный фрагмент, выделенный звёздочками внутри предложения). Это обеспечивает корректную работу метрики `has_answers` в `evaluate_task_relevance` без модификаций последней.

## 1. Установка зависимостей

In [60]:
!pip install -q pandas ujson spacy
!python -m spacy download en_core_web_sm -q



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 2. Импорты

In [61]:
import json
import random
import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import spacy
from datetime import datetime

nlp = spacy.load("en_core_web_sm")
random.seed(42)
np.random.seed(42)


## 3. Загрузка датасета EXPECT

Загружается обучающая часть датасета. На её основе строится словарь профилей ошибок: для каждой из 15 грамматических категорий доступен пул аннотированных примеров, из которых будут собираться задания для упражнений.

In [62]:
!git clone -q https://github.com/lorafei/Explainable_GEC.git


fatal: destination path 'Explainable_GEC' already exists and is not an empty directory.


In [63]:
DATA_PATH = Path("Explainable_GEC/data/json/train.json")

rows = []
with DATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
print(f"Записей в обучающей части EXPECT: {len(df)}")
print(f"Колонки: {list(df.columns)}")


Записей в обучающей части EXPECT: 15187
Колонки: ['target', 'source', 'evidence_index', 'correction_index', 'error_type', 'predicted_parsing_order', 'origin']


## 4. Базовые утилиты

Функция `detokenize` собирает токенизированное предложение в строку с корректной пунктуацией. Функция `analyze_correction_type` определяет, является ли исправление вставкой, удалением или заменой. Функция `clean_tokens` удаляет служебный токен `[NONE]`.

In [64]:
PUNCT_NO_SPACE_BEFORE = {".", ",", ":", ";", "!", "?", ")", "]", "}", "'s", "n't", "'re", "'ve", "'ll", "'d", "'m"}
PUNCT_NO_SPACE_AFTER = {"(", "[", "{", "$", "#"}

def clean_tokens(tokens: List[str]) -> List[str]:
    if not isinstance(tokens, list):
        return []
    return [t for t in tokens if t and t != "[NONE]"]

def detokenize(tokens: List[str]) -> str:
    tokens = clean_tokens(tokens)
    if not tokens:
        return ""
    out = []
    for i, tok in enumerate(tokens):
        if i == 0:
            out.append(tok)
        elif tok in PUNCT_NO_SPACE_BEFORE:
            out.append(tok)
        elif out and out[-1] in PUNCT_NO_SPACE_AFTER:
            out.append(tok)
        else:
            out.append(" " + tok)
    return "".join(out)

def analyze_correction_type(row: pd.Series) -> str:
    source_clean = clean_tokens(row["source"])
    target_clean = clean_tokens(row["target"])
    if len(target_clean) > len(source_clean):
        return "insertion"
    if len(target_clean) < len(source_clean):
        return "deletion"
    return "replacement"

df["source_text"] = df["source"].apply(detokenize)
df["target_text"] = df["target"].apply(detokenize)
df["correction_type"] = df.apply(analyze_correction_type, axis=1)
df["target_clean"] = df["target"].apply(clean_tokens)
df["source_clean"] = df["source"].apply(clean_tokens)
df["target_len"] = df["target_clean"].apply(len)

print("Типы исправлений:")
print(df["correction_type"].value_counts())
print("\nКатегории ошибок:")
print(df["error_type"].value_counts())


Типы исправлений:
correction_type
replacement    10063
insertion       3470
deletion        1654
Name: count, dtype: int64

Категории ошибок:
error_type
Preposition                     2079
Verb Tense                      2046
Number                          1728
Collocation                     1609
Others                          1294
POS Confusion                   1084
Subject-Verb Agreement          1016
Article                          873
Possessive                       846
Gerund                           799
Infinitives                      723
Auxiliary Verb                   332
Pronoun-Antecedent Agreement     325
Transitive Verb                  259
Participle                       174
Name: count, dtype: int64


## 5. Корректирующий токен и его позиция в target

Для replacement-исправлений правильный ответ находится в `target` на первой позиции из `correction_index`. Для insertion-исправлений правильный ответ — это вставленное слово, его позиция в `target` определяется тем же способом. Для deletion-исправлений шаблонный FiB не применим (нечего заменять на пропуск), такие записи исключаются из пула.

Поле `wrong_answer` хранит оригинальное слово ученика (для replacement) или маркер пропуска (для insertion). Это будущий ключевой дистрактор: упражнение должно тренировать именно ту ошибку, которую совершил ученик.

In [65]:
def remap_token_index(original_tokens: List[str], original_index: int) -> Optional[int]:
    """Map an index from raw EXPECT tokens (with [NONE]) into the cleaned token list."""
    if not isinstance(original_tokens, list):
        return None
    if not isinstance(original_index, int) or original_index < 0 or original_index >= len(original_tokens):
        return None
    if original_tokens[original_index] == "[NONE]":
        return None
    return sum(1 for t in original_tokens[:original_index] if t != "[NONE]")

def extract_correction(row: pd.Series) -> Dict[str, Any]:
    result = {"applicable": False, "target_word": None, "target_pos_idx": None, "wrong_word": None, "wrong_pos_idx": None}
    ctype = row["correction_type"]
    cidx = row["correction_index"]
    target_orig = row["target"]
    source_orig = row["source"]
    target_clean = row["target_clean"]
    source_clean = row["source_clean"]
    if not cidx or not isinstance(cidx, list) or len(cidx) == 0:
        return result
    pos_orig = cidx[0]
    if not isinstance(pos_orig, int):
        return result
    pos = remap_token_index(target_orig, pos_orig)
    if pos is None or pos >= len(target_clean):
        return result
    if ctype == "deletion":
        return result
    target_word = target_clean[pos]
    if target_word in PUNCT_NO_SPACE_BEFORE or target_word in PUNCT_NO_SPACE_AFTER:
        return result
    result["applicable"] = True
    result["target_word"] = target_word
    result["target_pos_idx"] = pos
    if ctype == "replacement":
        ev = row["evidence_index"]
        if isinstance(ev, list) and len(ev) > 0 and isinstance(ev[0], int):
            ev_pos = remap_token_index(source_orig, ev[0])
            if ev_pos is not None and ev_pos < len(source_clean):
                wrong = source_clean[ev_pos]
                if wrong and wrong != target_word and wrong != "[NONE]":
                    result["wrong_word"] = wrong
                    result["wrong_pos_idx"] = ev_pos
    elif ctype == "insertion":
        result["wrong_word"] = ""
    return result

extracted = df.apply(extract_correction, axis=1)
df["fib_applicable"] = extracted.apply(lambda x: x["applicable"])
df["fib_target_word"] = extracted.apply(lambda x: x["target_word"])
df["fib_target_pos"] = extracted.apply(lambda x: x["target_pos_idx"])
df["fib_wrong_word"] = extracted.apply(lambda x: x["wrong_word"])
df["fib_wrong_pos"] = extracted.apply(lambda x: x["wrong_pos_idx"])

print(f"Применимо к FiB: {df['fib_applicable'].sum()} из {len(df)} ({df['fib_applicable'].mean()*100:.1f}%)")


Применимо к FiB: 13336 из 15187 (87.8%)


## 6. POS-теггинг в синтаксическом контексте

Часть речи слова определяется не для изолированного токена, а для конкретной позиции в полном предложении. Это критически важно для корректности дистракторов: слово `to` в `"want to go"` будет PART, в `"to the store"` — ADP. spaCy-разбор предложения целиком даёт корректный POS-тег для каждой позиции.

Для оптимизации применяется кэширование разбора (`@functools.lru_cache`) и пакетная обработка через `nlp.pipe`.

In [66]:
import functools

POS_MAP = {
    "ADP": "PREP",
    "DET": "DET",
    "PRON": "PRON",
    "AUX": "AUX",
    "VERB": "VERB",
    "NOUN": "NOUN",
    "PROPN": "NOUN",
    "ADJ": "ADJ",
    "ADV": "ADV",
    "CCONJ": "CONJ",
    "SCONJ": "CONJ",
    "PART": "PART",
    "NUM": "NUM",
    "INTJ": "INTJ",
    "X": "OTHER",
    "SYM": "OTHER",
    "PUNCT": "PUNCT",
    "SPACE": "OTHER",
}

@functools.lru_cache(maxsize=50_000)
def _cached_doc_pos(sentence_text: str) -> Tuple[Tuple[str, str], ...]:
    """Cache (text_lower, mapped_pos) per spaCy token for a sentence."""
    if not sentence_text:
        return tuple()
    try:
        doc = nlp(sentence_text)
    except Exception:
        return tuple()
    return tuple((tok.text.lower(), POS_MAP.get(tok.pos_, "OTHER")) for tok in doc)

def pos_in_context(sentence_text: str, word: str, occurrence: int = 0) -> str:
    """POS of the `occurrence`-th match of `word` in `sentence_text` (spaCy-tagged in context)."""
    if not sentence_text or not word:
        return "OTHER"
    target_lower = word.lower()
    matches = [pos for text, pos in _cached_doc_pos(sentence_text) if text == target_lower]
    if not matches:
        return "OTHER"
    return matches[min(occurrence, len(matches) - 1)]

def pos_at_position(sentence_text: str, target_word: str) -> str:
    return pos_in_context(sentence_text, target_word, 0)

@functools.lru_cache(maxsize=200_000)
def pos_isolated(word: str) -> str:
    if not word:
        return "OTHER"
    try:
        doc = nlp(word)
    except Exception:
        return "OTHER"
    if len(doc) == 0:
        return "OTHER"
    return POS_MAP.get(doc[0].pos_, "OTHER")


## 7. Пул дистракторов

Дистракторы собираются исключительно из реальных ошибочных слов учеников в EXPECT. Никаких hard-coded списков частотных предлогов. Для каждой пары (error_type, POS) формируется множество слов, которые реальные ученики использовали ошибочно в этой категории. Это даёт педагогически содержательные дистракторы.

Дополнительно сохраняется частотность каждого слова в пуле — при выборе дистракторов предпочтение отдаётся более частотным ошибкам, что соответствует логике item writing guidelines: дистрактор должен отражать типичные заблуждения ученика.

In [67]:
def build_distractor_pool(df: pd.DataFrame) -> Dict[Tuple[str, str], Dict[str, int]]:
    pool: Dict[Tuple[str, str], Dict[str, int]] = defaultdict(lambda: defaultdict(int))
    valid = df[df["fib_applicable"] & df["fib_wrong_word"].notna() & (df["fib_wrong_word"] != "")]
    for _, row in valid.iterrows():
        error_type = row["error_type"]
        wrong = row["fib_wrong_word"]
        if not isinstance(wrong, str) or not wrong:
            continue
        if wrong in PUNCT_NO_SPACE_BEFORE or wrong in PUNCT_NO_SPACE_AFTER:
            continue
        source_clean = row["source_clean"]
        wrong_pos = row["fib_wrong_pos"]
        pos = "OTHER"
        if wrong_pos is not None and not (isinstance(wrong_pos, float) and pd.isna(wrong_pos)):
            try:
                wrong_pos_int = int(wrong_pos)
                occurrence = sum(1 for t in source_clean[:wrong_pos_int] if isinstance(t, str) and t.lower() == wrong.lower())
                pos = pos_in_context(row["source_text"], wrong, occurrence)
            except (TypeError, ValueError):
                pos = "OTHER"
        if pos == "OTHER":
            pos = pos_in_context(row["source_text"], wrong, 0)
        if pos == "OTHER":
            pos = pos_isolated(wrong)
        if pos in {"PUNCT", "OTHER"}:
            continue
        pool[(error_type, pos)][wrong] += 1
    return {k: dict(v) for k, v in pool.items()}

DISTRACTOR_POOL = build_distractor_pool(df)
print(f"Размер пула: {len(DISTRACTOR_POOL)} пар (error_type, POS)")
for (et, pos), words in list(DISTRACTOR_POOL.items())[:5]:
    top = sorted(words.items(), key=lambda x: -x[1])[:5]
    print(f"  {et} / {pos}: {top}")


Размер пула: 140 пар (error_type, POS)
  Subject-Verb Agreement / DET: [('the', 116), ('a', 61), ('The', 51), ('an', 15), ('some', 15)]
  Subject-Verb Agreement / ADJ: [('public', 22), ('Public', 9), ('many', 9), ('most', 3), ('Many', 3)]
  POS Confusion / DET: [('a', 48), ('the', 34), ('an', 9), ('The', 6), ('this', 5)]
  Auxiliary Verb / VERB: [('hope', 3), ('want', 1), ('think', 1), ('Imagine', 1), ('told', 1)]
  Preposition / DET: [('the', 132), ('a', 33), ('this', 10), ('The', 7), ('an', 7)]


## 8. Подбор дистракторов

Алгоритм подбора:
1. Определяется POS правильного ответа **в контексте** target-предложения.
2. Из пула берутся слова той же error_type и того же POS, отсортированные по убыванию частоты.
3. Главный дистрактор — `wrong_word` (если он есть и не совпадает с правильным ответом).
4. Регистр всех дистракторов приводится к регистру позиции пропуска (заглавная буква, если пропуск стоит в начале предложения).
5. Если не хватает кандидатов, добавляются слова той же POS из других error_type (общеграмматические дистракторы).

In [68]:
def match_case(distractor: str, target: str, is_sentence_start: bool) -> str:
    if not distractor:
        return distractor
    if is_sentence_start:
        return distractor[0].upper() + distractor[1:]
    if target and target[0].islower():
        return distractor[0].lower() + distractor[1:]
    return distractor

def get_distractors(
    target_word: str,
    error_type: str,
    target_pos_context: str,
    wrong_word: Optional[str],
    is_sentence_start: bool,
    n: int = 3,
    forbidden_lower: Optional[set] = None,
) -> List[str]:
    chosen: List[str] = []
    seen = {target_word.lower()}
    forbidden = set(forbidden_lower) if forbidden_lower else set()
    if not isinstance(wrong_word, str):
        wrong_word = None
    if wrong_word and wrong_word.lower() not in seen and wrong_word != "" and wrong_word.lower() not in forbidden:
        chosen.append(match_case(wrong_word, target_word, is_sentence_start))
        seen.add(wrong_word.lower())
    primary = DISTRACTOR_POOL.get((error_type, target_pos_context), {})
    candidates = sorted(primary.items(), key=lambda x: -x[1])
    for word, _ in candidates:
        if len(chosen) >= n:
            break
        if word.lower() in seen or word.lower() in forbidden:
            continue
        chosen.append(match_case(word, target_word, is_sentence_start))
        seen.add(word.lower())
    if len(chosen) < n:
        for (et, pos), words in DISTRACTOR_POOL.items():
            if pos != target_pos_context:
                continue
            if et == error_type:
                continue
            for word, _ in sorted(words.items(), key=lambda x: -x[1]):
                if len(chosen) >= n:
                    break
                if word.lower() in seen or word.lower() in forbidden:
                    continue
                chosen.append(match_case(word, target_word, is_sentence_start))
                seen.add(word.lower())
            if len(chosen) >= n:
                break
    if len(chosen) == 0 and wrong_word == "":
        chosen = ["(no word needed)"]
    return chosen[:n]


## 9. Карта инструкций и типов заданий по error_type

Каждая грамматическая категория EXPECT соотносится с подходящим типом задания и шаблонной инструкцией. Тип задания берётся из таксономии, согласованной с дообученными LLM (`grammar_choice` / `vocabulary_fill` / `transformation`). Инструкции содержательны для преподавателя-эксперта и явно указывают на категорию ошибки, которая тренируется.

In [69]:
ERROR_TYPE_CONFIG: Dict[str, Dict[str, str]] = {
    "Preposition":              {"fib_type": "grammar_choice",  "instruction": "Choose the correct preposition to complete each sentence."},
    "Article":                  {"fib_type": "grammar_choice",  "instruction": "Choose the correct article (a / an / the / —) for each sentence."},
    "Verb Tense":               {"fib_type": "grammar_choice",  "instruction": "Choose the correct verb tense to complete each sentence."},
    "Subject-Verb Agreement":   {"fib_type": "grammar_choice",  "instruction": "Choose the verb form that agrees with the subject."},
    "Auxiliary Verb":           {"fib_type": "grammar_choice",  "instruction": "Choose the correct auxiliary verb for each sentence."},
    "Pronoun-Antecedent Agreement": {"fib_type": "grammar_choice", "instruction": "Choose the pronoun that agrees with its antecedent."},
    "Transitive Verb":          {"fib_type": "grammar_choice",  "instruction": "Choose the correct transitive verb for each sentence."},
    "Gerund":                   {"fib_type": "grammar_choice",  "instruction": "Choose the correct gerund form for each sentence."},
    "Infinitives":              {"fib_type": "grammar_choice",  "instruction": "Choose the correct infinitive form for each sentence."},
    "Participle":               {"fib_type": "grammar_choice",  "instruction": "Choose the correct participle form for each sentence."},
    "Number":                   {"fib_type": "grammar_choice",  "instruction": "Choose the correct number form (singular / plural) for each sentence."},
    "Possessive":               {"fib_type": "grammar_choice",  "instruction": "Choose the correct possessive form for each sentence."},
    "POS Confusion":            {"fib_type": "vocabulary_fill", "instruction": "Choose the word with the correct part of speech to complete each sentence."},
    "Collocation":              {"fib_type": "vocabulary_fill", "instruction": "Choose the word that collocates correctly in each sentence."},
    "Others":                   {"fib_type": "grammar_choice",  "instruction": "Choose the correct word to complete each sentence."},
}

def config_for(error_type: str) -> Dict[str, str]:
    return ERROR_TYPE_CONFIG.get(error_type, ERROR_TYPE_CONFIG["Others"])


## 10. Группировка записей в профили ошибок

Для каждой error_type формируется пул применимых записей. На этапе генерации упражнения из пула выбирается 8–12 записей (целевое количество — 10), которые становятся items одного упражнения. Целевое число items соответствует диапазону, заданному в `ITEM_COUNT_RANGES` для дообученных LLM, что обеспечивает прямое соответствие метрики `items_count_ok`.

In [70]:
TARGET_ITEMS = 10
MIN_ITEMS = 8
MAX_ITEMS = 12

applicable_df = df[df["fib_applicable"]].copy()
profiles: Dict[str, pd.DataFrame] = {
    et: g.reset_index(drop=True) for et, g in applicable_df.groupby("error_type")
}
for et, g in profiles.items():
    print(f"{et}: {len(g)} применимых записей")


Article: 545 применимых записей
Auxiliary Verb: 293 применимых записей
Collocation: 1414 применимых записей
Gerund: 663 применимых записей
Infinitives: 678 применимых записей
Number: 1719 применимых записей
Others: 944 применимых записей
POS Confusion: 967 применимых записей
Participle: 168 применимых записей
Possessive: 687 применимых записей
Preposition: 1936 применимых записей
Pronoun-Antecedent Agreement: 320 применимых записей
Subject-Verb Agreement: 1011 применимых записей
Transitive Verb: 178 применимых записей
Verb Tense: 1813 применимых записей


## 11. Fill-in-the-Blanks v2

Для каждого item:
1. Строится template — target-предложение с заменённым на `___` корректирующим токеном.
2. Определяется POS правильного ответа в контексте полного target-предложения.
3. Подбираются 3 дистрактора, включая `wrong_word` ученика как ключевой.
4. Опции перемешиваются.
5. В `student_answer_en` правильный токен помечается двойными звёздочками `**word**` — формат, эквивалентный тому, что используется в разметке Qwen3-235B для обучения LLM.

Поле `corrected_sentence` упражнения берётся из первого item — оно используется метрикой Error Cat как маркер того, что метод знает корректное предложение.

In [71]:
def build_template(target_clean: List[str], blank_pos: int) -> str:
    tokens = list(target_clean)
    if blank_pos < 0 or blank_pos >= len(tokens):
        return ""
    tokens[blank_pos] = "___"
    return detokenize(tokens)

def build_student_answer(target_clean: List[str], blank_pos: int) -> str:
    tokens = list(target_clean)
    if blank_pos < 0 or blank_pos >= len(tokens):
        return ""
    tokens[blank_pos] = f"**{tokens[blank_pos]}**"
    return detokenize(tokens)

def build_fib_item(row: pd.Series, target_pos_context: str) -> Optional[Dict[str, Any]]:
    target_clean = row["target_clean"]
    blank_pos = row["fib_target_pos"]
    target_word = row["fib_target_word"]
    if not isinstance(target_clean, list) or blank_pos is None or not target_word:
        return None
    try:
        blank_pos = int(blank_pos)
    except (TypeError, ValueError):
        return None
    template = build_template(target_clean, blank_pos)
    if not template:
        return None
    is_start = blank_pos == 0
    template_words_lower = {t.lower() for t in target_clean if isinstance(t, str)}
    template_words_lower.discard(target_word.lower())
    distractors = get_distractors(
        target_word=target_word,
        error_type=row["error_type"],
        target_pos_context=target_pos_context,
        wrong_word=row["fib_wrong_word"],
        is_sentence_start=is_start,
        n=3,
        forbidden_lower=template_words_lower,
    )
    if len(distractors) < 2:
        return None
    correct_display = target_word
    if is_start:
        correct_display = correct_display[0].upper() + correct_display[1:]
    options = [correct_display] + distractors
    random.shuffle(options)
    student_answer = build_student_answer(target_clean, blank_pos)
    return {
        "question_en": template,
        "options_en": options,
        "student_answer_en": student_answer,
    }

def generate_fib_exercise(error_type: str, sample_pool: pd.DataFrame, n_items: int = TARGET_ITEMS) -> Optional[Dict[str, Any]]:
    if len(sample_pool) < MIN_ITEMS:
        return None
    n_items = max(MIN_ITEMS, min(MAX_ITEMS, n_items))
    cfg = config_for(error_type)
    items: List[Dict[str, Any]] = []
    corrected_sentence_ref: Optional[str] = None
    iterator = sample_pool.sample(frac=1.0, random_state=random.randint(0, 10_000)).iterrows()
    for _, row in iterator:
        if len(items) >= n_items:
            break
        target_word = row["fib_target_word"]
        sentence_text = row["target_text"]
        target_clean = row["target_clean"]
        blank_pos = row["fib_target_pos"]
        try:
            blank_pos_int = int(blank_pos)
        except (TypeError, ValueError):
            continue
        if not isinstance(target_clean, list) or blank_pos_int < 0 or blank_pos_int >= len(target_clean):
            continue
        occurrence = sum(1 for t in target_clean[:blank_pos_int] if isinstance(t, str) and t.lower() == target_word.lower())
        target_pos_context = pos_in_context(sentence_text, target_word, occurrence)
        if target_pos_context in {"OTHER", "PUNCT"}:
            target_pos_context = pos_isolated(target_word)
        item = build_fib_item(row, target_pos_context)
        if item is None:
            continue
        items.append(item)
        if corrected_sentence_ref is None:
            corrected_sentence_ref = sentence_text
    if len(items) < MIN_ITEMS:
        return None
    return {
        "target_error_category": error_type,
        "corrected_sentence": corrected_sentence_ref or "",
        "task": {
            "type": cfg["fib_type"],
            "instruction_en": cfg["instruction"],
            "content_en": {"items": items},
        },
    }


## 12. Sentence Reconstruction v2

Прежний жёсткий фильтр `4 ≤ len ≤ 20` снят. Остаётся только нижняя граница: предложение должно содержать минимум 4 значимых токена, иначе его не имеет смысла восстанавливать. Верхняя граница отсутствует — предложения любой длины обрабатываются, длинные через chunking.

Алгоритм:
1. Предложение разбивается на синтаксические фрагменты по запятым, точкам с запятой, тире и двоеточиям. Конечная пунктуация (`.!?`) приклеивается к последнему фрагменту.
2. Если в предложении нет внутренней пунктуации и оно длиннее 14 токенов — оно дополнительно разбивается на чанки фиксированной длины 6 токенов по границам слов.
3. Полученные чанки (от 2 до 6 на предложение) перемешиваются как неделимые блоки. Внутренний порядок слов внутри чанка сохраняется.
4. Если получился ровно один чанк (короткое простое предложение длиной 4–14 токенов), применяется классическое перемешивание отдельных токенов с фиксацией первого и последнего значимого слова.

Покрытие поднимается с 39% до 95%+, поскольку большинство предложений EXPECT длиннее 20 токенов и теперь обрабатываются через chunking.

In [72]:
PUNCT_CHUNK_SPLIT = {",", ";", ":"}
PUNCT_DASH = {"--", "---", "–", "—"}
PUNCT_TERMINAL = {".", "!", "?"}
LONG_CHUNK_THRESHOLD = 12
SUB_CHUNK_SIZE = 6

def split_into_chunks(tokens: List[str]) -> List[List[str]]:
    chunks: List[List[str]] = []
    current: List[str] = []
    for tok in tokens:
        if tok in PUNCT_CHUNK_SPLIT or tok in PUNCT_DASH:
            if current:
                current.append(tok)
                chunks.append(current)
                current = []
        elif tok in PUNCT_TERMINAL:
            current.append(tok)
        else:
            current.append(tok)
    if current:
        chunks.append(current)
    return [c for c in chunks if any(t not in PUNCT_NO_SPACE_BEFORE for t in c)]

def chunk_by_length(tokens: List[str], chunk_size: int = SUB_CHUNK_SIZE) -> List[List[str]]:
    chunks: List[List[str]] = []
    i = 0
    n = len(tokens)
    while i < n:
        end = min(i + chunk_size, n)
        chunks.append(tokens[i:end])
        i = end
    return chunks

def merge_punct_only_chunks(chunks: List[List[str]]) -> List[List[str]]:
    """Merge chunks lacking any meaningful token into the previous chunk to avoid orphan punctuation fragments."""
    out: List[List[str]] = []
    for c in chunks:
        if any(is_meaningful(t) for t in c):
            out.append(list(c))
        elif out:
            out[-1].extend(c)
        else:
            out.append(list(c))
    return out

def split_long_chunks(chunks: List[List[str]], threshold: int = LONG_CHUNK_THRESHOLD, chunk_size: int = SUB_CHUNK_SIZE) -> List[List[str]]:
    out: List[List[str]] = []
    for c in chunks:
        if len(c) > threshold:
            out.extend(chunk_by_length(c, chunk_size=chunk_size))
        else:
            out.append(c)
    return merge_punct_only_chunks(out)

def is_meaningful(token: str) -> bool:
    return token not in PUNCT_NO_SPACE_BEFORE and token not in PUNCT_NO_SPACE_AFTER and token not in PUNCT_DASH

def chunks_to_text(chunks: List[List[str]]) -> str:
    flat: List[str] = []
    for c in chunks:
        flat.extend(c)
    return detokenize(flat)

def shuffle_chunks(chunks: List[List[str]], seed: int) -> List[List[str]]:
    if len(chunks) <= 1:
        return chunks
    rng = random.Random(seed)
    order = list(range(len(chunks)))
    for _ in range(10):
        rng.shuffle(order)
        if order != list(range(len(chunks))):
            break
    return [chunks[i] for i in order]

def shuffle_short_sentence(tokens: List[str], seed: int) -> List[str]:
    meaningful_idx = [i for i, t in enumerate(tokens) if is_meaningful(t)]
    if len(meaningful_idx) < 3:
        return list(tokens)
    rng = random.Random(seed)
    inner = meaningful_idx[1:-1]
    inner_words = [tokens[i] for i in inner]
    for _ in range(10):
        shuffled_inner = inner_words[:]
        rng.shuffle(shuffled_inner)
        if shuffled_inner != inner_words:
            break
    out = list(tokens)
    for idx, new_word in zip(inner, shuffled_inner):
        out[idx] = new_word
    return out

def build_reconstruction_item(row: pd.Series, seed: int) -> Optional[Dict[str, Any]]:
    target_clean = row["target_clean"]
    if not isinstance(target_clean, list) or len(target_clean) < 4:
        return None
    chunks = split_into_chunks(target_clean)
    if len(chunks) == 0:
        return None
    if len(chunks) == 1 and len(chunks[0]) <= 14:
        only = chunks[0]
        shuffled_tokens = shuffle_short_sentence(only, seed)
        if shuffled_tokens == only:
            return None
        question = " / ".join(shuffled_tokens)
        answer = detokenize(only)
        return {
            "question_en": f"Restore the correct word order: {question}",
            "options_en": [],
            "student_answer_en": f"**{answer}**",
        }
    chunks = split_long_chunks(chunks)
    if len(chunks) < 2:
        return None
    shuffled = shuffle_chunks(chunks, seed)
    if shuffled == chunks:
        return None
    question_fragments = [detokenize(c) for c in shuffled]
    question = " / ".join(question_fragments)
    answer = chunks_to_text(chunks)
    return {
        "question_en": f"Restore the correct order of fragments: {question}",
        "options_en": [],
        "student_answer_en": f"**{answer}**",
    }

def generate_recon_exercise(error_type: str, sample_pool: pd.DataFrame, n_items: int = TARGET_ITEMS) -> Optional[Dict[str, Any]]:
    if len(sample_pool) < MIN_ITEMS:
        return None
    n_items = max(MIN_ITEMS, min(MAX_ITEMS, n_items))
    items: List[Dict[str, Any]] = []
    corrected_sentence_ref: Optional[str] = None
    seed_base = random.randint(0, 100_000)
    for k, (_, row) in enumerate(sample_pool.sample(frac=1.0, random_state=random.randint(0, 10_000)).iterrows()):
        if len(items) >= n_items:
            break
        item = build_reconstruction_item(row, seed=seed_base + k)
        if item is None:
            continue
        items.append(item)
        if corrected_sentence_ref is None:
            corrected_sentence_ref = row["target_text"]
    if len(items) < MIN_ITEMS:
        return None
    return {
        "target_error_category": error_type,
        "corrected_sentence": corrected_sentence_ref or "",
        "task": {
            "type": "transformation",
            "instruction_en": f"Restore the correct order in each sentence. Focus on the structure typical for {error_type.lower()} corrections.",
            "content_en": {"items": items},
        },
    }


## 13. Генерация по валидационной выборке val_300

Валидационная выборка `val_meta_300.json` хранит для каждого примера error_type и task_type — те же входные параметры, по которым отвечают дообученные LLM. Для каждого примера val_300 каждый шаблонный метод формирует одно упражнение, используя пул записей из EXPECT-train той же error_type.

Этот сценарий имитирует production-пайплайн SAYIT: на вход поступает профиль ошибок ученика (error_type), на выход — одно упражнение с 8–12 заданиями. Сравнение с LLM становится прямым.

In [73]:
VAL_META_PATH_CANDIDATES = [
    Path("data/val_meta.json"),
    Path("../data/val_meta.json"),
    Path("val_meta.json"),
    Path("val_meta_300.json"),
]
VAL_META_PATH = next((p for p in VAL_META_PATH_CANDIDATES if p.exists()), VAL_META_PATH_CANDIDATES[0])

if VAL_META_PATH.exists():
    with VAL_META_PATH.open("r", encoding="utf-8") as f:
        val_meta = json.load(f)
    print(f"Загружено {len(val_meta)} мета-записей из {VAL_META_PATH}")
else:
    val_meta = []
    print("val_meta.json не найден — загрузите его перед запуском следующих ячеек")


Загружено 300 мета-записей из ../data/val_meta.json


In [74]:
def generate_all_for_val(method_fn, val_meta: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    results: List[Dict[str, Any]] = []
    for meta in val_meta:
        error_type = meta.get("error_type") or meta.get("target_error_category") or "Others"
        if error_type not in profiles:
            error_type = "Others" if "Others" in profiles else next(iter(profiles))
        pool = profiles[error_type]
        exercise = method_fn(error_type, pool)
        if exercise is None:
            exercise = {
                "target_error_category": error_type,
                "corrected_sentence": "",
                "task": {
                    "type": "grammar_choice",
                    "instruction_en": "",
                    "content_en": {"items": []},
                },
            }
        results.append(exercise)
    return results

if val_meta:
    random.seed(42)
    fib_predictions = generate_all_for_val(generate_fib_exercise, val_meta)
    random.seed(42)
    recon_predictions = generate_all_for_val(generate_recon_exercise, val_meta)
    print(f"FiB: {len(fib_predictions)} упражнений")
    print(f"Recon: {len(recon_predictions)} упражнений")


FiB: 300 упражнений
Recon: 300 упражнений


## 14. Примеры сгенерированных упражнений

Ниже выводятся по одному упражнению каждого метода для визуальной проверки качества.

In [75]:
def show_exercise(title: str, exercise: Dict[str, Any], max_items: int = 3) -> None:
    print("=" * 70)
    print(title)
    print("=" * 70)
    print(f"target_error_category: {exercise['target_error_category']}")
    print(f"task.type:             {exercise['task']['type']}")
    print(f"instruction:           {exercise['task']['instruction_en']}")
    items = exercise['task']['content_en']['items']
    print(f"items ({len(items)}):")
    for i, it in enumerate(items[:max_items], 1):
        print(f"  {i}. Q: {it['question_en']}")
        if it['options_en']:
            print(f"     options: {it['options_en']}")
        print(f"     answer:  {it['student_answer_en']}")
    if len(items) > max_items:
        print(f"     ... и ещё {len(items) - max_items} item(s)")
    print()

if val_meta:
    show_exercise("Fill-in-the-Blanks v2", fib_predictions[0])
    show_exercise("Sentence Reconstruction v2", recon_predictions[0])


Fill-in-the-Blanks v2
target_error_category: Preposition
task.type:             grammar_choice
instruction:           Choose the correct preposition to complete each sentence.
items (10):
  1. Q: This means that a number of offenders can not sustain their livelihood, and they desperately experience limited options ___ earning money.
     options: ['in', 'from', 'with', 'for']
     answer:  This means that a number of offenders can not sustain their livelihood, and they desperately experience limited options **for** earning money.
  2. Q: I really like many sports ___ this world. I go to the gym daily to stay fit and healthy.
     options: ['with', 'for', 'in', 'from']
     answer:  I really like many sports **in** this world. I go to the gym daily to stay fit and healthy.
  3. Q: If you want to take up swimming, I would recommend you to do your best and if you like it you will enjoy it at the same time ___ you are doing exercise.
     options: ['when', 'before', 'because', 'as']
     a

## 15. Coverage по методам

Coverage — доля примеров `val_300`, для которых метод смог сгенерировать упражнение с количеством items в допустимом диапазоне (8–12).

In [76]:
def coverage(predictions: List[Dict[str, Any]]) -> float:
    if not predictions:
        return 0.0
    ok = 0
    for p in predictions:
        items = p.get("task", {}).get("content_en", {}).get("items", [])
        if MIN_ITEMS <= len(items) <= MAX_ITEMS:
            ok += 1
    return ok / len(predictions)

if val_meta:
    print(f"Coverage FiB:   {coverage(fib_predictions)*100:.1f}%")
    print(f"Coverage Recon: {coverage(recon_predictions)*100:.1f}%")


Coverage FiB:   100.0%
Coverage Recon: 100.0%


## 16. Сериализация предсказаний для метрик

Предсказания сохраняются в формате, совместимом с `evaluate_task_relevance` из ноутбука `eval_300.ipynb`: список строк, где каждая строка — JSON-сериализация одного упражнения. Эти файлы загружаются как `all_predictions[name]` рядом с предсказаниями LLM.

In [77]:
def dump_predictions(predictions: List[Dict[str, Any]], path: str) -> None:
    serialized = [json.dumps(p, ensure_ascii=False) for p in predictions]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serialized, f, ensure_ascii=False, indent=2)
    print(f"Сохранено {len(serialized)} предсказаний → {path}")

if val_meta:
    dump_predictions(fib_predictions, "baseline_fib_v2_predictions.json")
    dump_predictions(recon_predictions, "baseline_recon_v2_predictions.json")


Сохранено 300 предсказаний → baseline_fib_v2_predictions.json
Сохранено 300 предсказаний → baseline_recon_v2_predictions.json


## 17. Интеграция в evaluate_task_relevance

В ноутбуке `eval_300.ipynb` достаточно добавить два новых имени в список методов и загрузить соответствующие JSON-файлы:

```python
BASELINE_MODELS = ["baseline_fib_v2", "baseline_recon_v2"]

for name, path in [
    ("baseline_fib_v2", "baseline_fib_v2_predictions.json"),
    ("baseline_recon_v2", "baseline_recon_v2_predictions.json"),
]:
    with open(path, "r", encoding="utf-8") as f:
        all_predictions[name] = json.load(f)
```

В `ITEM_COUNT_RANGES` для типов задания `grammar_choice`, `vocabulary_fill` и `transformation` значения уже определены, правки маппинга не требуются. Подметрики `json_valid`, `task_type_match`, `error_cat_present`, `items_count_ok`, `has_answers` рассчитываются единообразно для шаблонов и LLM, что обеспечивает методологически корректное сравнение.

## 18. Сохранение в общий файл предсказаний

Оба метода сохраняются в `data/all_predictions_300_v2.json` под ключами `baseline_fib_v2` и `baseline_recon_v2` рядом с предсказаниями LLM (`qwen2.5-3b`, `smollm3-3b`, `phi3.5-mini`). Значения — списки из 300 JSON-сериализованных упражнений, как и у моделей. Существующие ключи в файле сохраняются.

In [78]:
OUT_PATH_CANDIDATES = [
    Path("data/all_predictions_300_v2.json"),
    Path("../data/all_predictions_300_v2.json"),
    Path("all_predictions_300_v2.json"),
]
out_path = next((p for p in OUT_PATH_CANDIDATES if p.parent.exists()), OUT_PATH_CANDIDATES[-1])

existing: Dict[str, List[str]] = {}
if out_path.exists():
    with out_path.open("r", encoding="utf-8") as f:
        existing = json.load(f)

existing["baseline_fib_v2"] = [json.dumps(p, ensure_ascii=False) for p in fib_predictions]
existing["baseline_recon_v2"] = [json.dumps(p, ensure_ascii=False) for p in recon_predictions]

with out_path.open("w", encoding="utf-8") as f:
    json.dump(existing, f, ensure_ascii=False, indent=2)

print(f"Сохранено в {out_path}")
for k, v in existing.items():
    print(f"  {k}: {len(v)} упражнений")

Сохранено в ../data/all_predictions_300_v2.json
  qwen2.5-3b: 300 упражнений
  smollm3-3b: 300 упражнений
  phi3.5-mini: 300 упражнений
  baseline_fib_v2: 300 упражнений
  baseline_recon_v2: 300 упражнений
